# LFS Reporting Framework — Demo Runbook

This notebook walks through the complete **Loss Forecast Suite (LFS)** reporting
pipeline using **fully synthetic mock data**. No production tables are read or written.

### What this notebook covers

| Layer | Purpose | Key outputs |
|---|---|---|
| **Layer 1** | Account-level enrichment | Deciles, bands, buckets, DQ flags |
| **Layer 2** | Aggregated business tables | Summaries by channel, source, line, bucket |
| **Layer 3** | Model monitoring | Score drift, feature PSI, data quality, population mix |

### How to connect to real data
Replace the mock-data cells with a call to `runner.run()` (see `docs/notebook_runbook.md`)
once you are ready to run against the production catalog.

## 1. Setup

In [ ]:
import os, sys
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17"
os.environ["PATH"] = f'{os.environ["JAVA_HOME"]}/bin:' + os.environ.get("PATH", "")

print("JAVA_HOME =", os.environ["JAVA_HOME"])
print("python =", sys.version)

# Resolve project root (one level above this notebooks/ directory).
NOTEBOOK_DIR = os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, ".."))

# Fallback: if running from project root directly.
if not os.path.isdir(os.path.join(PROJECT_ROOT, "framework")):
    PROJECT_ROOT = NOTEBOOK_DIR

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")
print(f"sys.path[0]:  {sys.path[0]}")

In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# In a DSW environment, `spark` is already available.
# Uncomment the block below only when running locally.
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("LFS_Reporting_Demo")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} ready")

In [ ]:
from framework.config import load_model_config
from framework.layer1 import enrich_layer1
from framework.layer2 import build_layer2
from framework.layer3 import build_layer3
from notebooks.mock_data import generate_lfs_mock_data

print("Framework imports OK")

## 2. Synthetic Mock Data

The `mock_data` module generates two DataFrames that match the LFS source schema:

- **`baseline_df`** — 7 months (2024-08 → 2025-02), 2 channels × 250 accounts/month  
  Used for static bin edges and Layer 3 baseline metrics.
- **`current_df`** — 3 months (2025-03 → 2025-05), 2 channels × 200 accounts/month  
  The scored population being reported on.

**Built-in drift signals** so monitoring tables are non-trivial:

| Signal | Description |
|---|---|
| Score mean +0.06–0.08 | Mild upward shift current vs baseline |
| Score std +0.03–0.04 | Slight tail thickening |
| `feature_03` mean +0.14 | Largest feature drift — should flag in feature PSI |
| `feature_01` mean +0.06 | Secondary feature drift |
| `feature_10` mean +0.08 | Moderate feature drift |
| Source mix shift | "organic" −10 pp, "paid"/"referral" each +5 pp |
| Channel difference | digital scores ~0.04–0.06 higher than directmail |

In [ ]:
data = generate_lfs_mock_data(
    spark,
    n_baseline_per_month_channel=250,
    n_current_per_month_channel=200,
    seed=42,
)
baseline_df = data["baseline_df"]
current_df  = data["current_df"]

In [ ]:
print("=== Baseline: score stats by channel ===")
(baseline_df
 .groupBy("channel")
 .agg(
     F.count("*").alias("n_accounts"),
     F.round(F.avg("lfs_score"), 4).alias("mean_score"),
     F.round(F.stddev("lfs_score"), 4).alias("std_score"),
     F.round(F.min("lfs_score"), 4).alias("min_score"),
     F.round(F.max("lfs_score"), 4).alias("max_score"),
 )
 .orderBy("channel")
 .show()
)

print("=== Current: score stats by (month, channel) ===")
(current_df
 .groupBy("month", "channel")
 .agg(
     F.count("*").alias("n_accounts"),
     F.round(F.avg("lfs_score"), 4).alias("mean_score"),
     F.round(F.stddev("lfs_score"), 4).alias("std_score"),
 )
 .orderBy("month", "channel")
 .show()
)

In [ ]:
print("=== Source mix: baseline vs current ===")
base_mix = (baseline_df
    .groupBy("source")
    .agg(F.count("*").alias("n"))
    .withColumn("pct", F.round(F.col("n") / baseline_df.count(), 3))
    .withColumn("period", F.lit("baseline"))
)
curr_mix = (current_df
    .groupBy("source")
    .agg(F.count("*").alias("n"))
    .withColumn("pct", F.round(F.col("n") / current_df.count(), 3))
    .withColumn("period", F.lit("current"))
)
base_mix.union(curr_mix).orderBy("period", "source").show()

## 3. Load LFS Configuration

The pipeline is entirely config-driven.  `load_model_config("lfs")` reads
`conf/models/lfs.yaml` and returns a typed `ModelConfig` object — no code
changes needed when tuning thresholds, adding features, or renaming tables.

In [ ]:
config = load_model_config("lfs")

print(f"Model         : {config.name} — {config.display_name}")
print(f"Channels      : {config.source['channels']}")
print(f"Score column  : {config.source['score_col']}")
print(f"Features      : {config.layer3['feature_columns']}")
print()

l2_tables = (
    list(config.layer2.get("standard_tables", {}).keys())
    + list(config.layer2.get("driver_tables", {}).keys())
)
print(f"Layer 2 tables ({len(l2_tables)}):")
for t in l2_tables:
    print(f"  {config.layer2.get('standard_tables', {}).get(t, config.layer2.get('driver_tables', {}).get(t, {})).get('output_table', t)}")

print()
print(f"Layer 3 tables ({len(config.layer3['tables'])}):")
for t, cfg in config.layer3["tables"].items():
    print(f"  {cfg['output_table']}")

## 4. Layer 1 — Account-Level Enrichment

`enrich_layer1` adds these columns to every account row:

| Column | Description |
|---|---|
| `score_month` | Pipeline run label (passed in as argument) |
| `model_version` | Version tag |
| `vintage_month` | Per-row copy of the scoring month |
| `lfs_decile_dyn` | Dynamic decile (1–10) computed within each (month, channel) |
| `lfs_band_dyn` | Decile band label: Low / Medium / High |
| `lfs_decile_static` | Static decile using edges from the baseline window |
| `receivable_bucket` | Fixed-boundary bucket for `endingreceivable` |
| `saleamount_bucket` | Fixed-boundary bucket for `saleamount` |
| `salecount_bucket` | Fixed-boundary bucket for `salecount` |
| `mob_pti_bucket` | Fixed-boundary bucket for `mob_pti` |
| `dq_missing_flag` | 1 if `lfs_score` is NULL |
| `dq_outlier_flag` | 1 if `lfs_score` outside [0, 1] |

The current data spans 3 months; Layer 1 enriches all of them in one pass.

In [ ]:
SCORE_MONTH   = "2025-05"   # pipeline-run label (latest current month)
MODEL_VERSION = "v1.0"

enriched_df = enrich_layer1(
    current_df,
    config,
    baseline_df,
    score_month=SCORE_MONTH,
    model_version=MODEL_VERSION,
)
print(f"Layer 1 complete — {enriched_df.count():,} rows, {len(enriched_df.columns)} columns")
print(f"New columns added by Layer 1:")
orig_cols = set(current_df.columns)
new_cols  = [c for c in enriched_df.columns if c not in orig_cols]
print("  " + ", ".join(new_cols))

In [ ]:
print("=== Layer 1 sample — enriched columns ===")
enriched_df.select(
    "creditaccountid", "vintage_month", "channel",
    "lfs_score",
    "lfs_decile_dyn", "lfs_band_dyn", "lfs_decile_static",
    "receivable_bucket", "saleamount_bucket",
    "dq_missing_flag", "dq_outlier_flag",
).show(15, truncate=False)

In [ ]:
from pyspark.sql import Window
print("=== Dynamic decile distribution per (vintage_month, channel) ===")
print("Each decile should contain ~10% of accounts within its partition.\n")
(enriched_df
 .groupBy("vintage_month", "channel", "lfs_decile_dyn")
 .agg(F.count("*").alias("n_accounts"))
 .withColumn(
     "pct",
     F.round(
         F.col("n_accounts") /
         F.sum("n_accounts").over(
             Window
             .partitionBy("vintage_month", "channel")
         ),
         3,
     )
 )
 .orderBy("vintage_month", "channel", "lfs_decile_dyn")
 .show(60)
)

In [ ]:
from pyspark.sql import Window
print("=== Band distribution per channel ===")
(enriched_df
 .groupBy("channel", "lfs_band_dyn")
 .agg(F.count("*").alias("n_accounts"))
 .withColumn(
     "pct_of_channel",
     F.round(
         F.col("n_accounts") /
         F.sum("n_accounts").over(
             Window
             .partitionBy("channel")
         ),
         3,
     )
 )
 .orderBy("channel", "lfs_band_dyn")
 .show()
)

## 5. Layer 2 — Aggregated Business Tables

`build_layer2` produces 8 tables from the enriched Layer 1 output.  All tables
keep `channel` as a column — no physical split.

**Standard tables** (4):

| Table | Group by |
|---|---|
| `lfs_layer2_overall_summary` | vintage_month, channel |
| `lfs_layer2_score_distribution` | vintage_month, channel, lfs_decile_dyn |
| `lfs_layer2_by_source` | vintage_month, channel, source |
| `lfs_layer2_by_line` | vintage_month, channel, line |

**Driver / drill-down tables** (4):

| Table | Group by |
|---|---|
| `lfs_layer2_by_receivable_bucket` | vintage_month, channel, receivable_bucket |
| `lfs_layer2_by_saleamount_bucket` | vintage_month, channel, saleamount_bucket |
| `lfs_layer2_by_salecount_bucket` | vintage_month, channel, salecount_bucket |
| `lfs_layer2_by_mob_pti_bucket` | vintage_month, channel, mob_pti_bucket |

In [ ]:
l2_tables = build_layer2(enriched_df, config)
print(f"Layer 2 complete — {len(l2_tables)} tables produced:")
for tname, tdf in l2_tables.items():
    print(f"  {tname:<45}  {tdf.count():>4} rows  {len(tdf.columns)} cols")

In [ ]:
print("=== Overall Summary (lfs_layer2_overall_summary) ===")
print("One row per (vintage_month, channel).  Metrics: count, avg/sum score, receivable, sale.\n")
(l2_tables["lfs_layer2_overall_summary"]
 .select(
     "vintage_month", "channel",
     "account_count",
     F.round("avg_lfs_score", 4).alias("avg_lfs_score"),
     F.round("avg_endingreceivable", 2).alias("avg_recv"),
     F.round("avg_saleamount", 2).alias("avg_saleamt"),
 )
 .orderBy("vintage_month", "channel")
 .show(truncate=False)
)

In [ ]:
print("=== Score Distribution (lfs_layer2_score_distribution) ===")
print("One row per (vintage_month, channel, decile).  pct_of_channel_vintage_accounts should ~= 0.10.\n")
(l2_tables["lfs_layer2_score_distribution"]
 .select(
     "vintage_month", "channel", "lfs_decile_dyn",
     "account_count",
     F.round("avg_lfs_score", 4).alias("avg_score"),
     F.round("pct_of_channel_vintage_accounts", 4).alias("pct"),
 )
 .orderBy("vintage_month", "channel", "lfs_decile_dyn")
 .show(60, truncate=False)
)

In [ ]:
print("=== By Source (lfs_layer2_by_source) ===")
print("Shows how average score and financials differ across acquisition sources.\n")
(l2_tables["lfs_layer2_by_source"]
 .select(
     "vintage_month", "channel", "source",
     "account_count",
     F.round("avg_lfs_score", 4).alias("avg_score"),
     F.round("avg_saleamount", 2).alias("avg_saleamt"),
 )
 .orderBy("vintage_month", "channel", "source")
 .show(truncate=False)
)

## 6. Layer 3 — Model Monitoring

`build_layer3` compares the current period against the baseline window.
Six monitoring tables are produced:

| Table | Rows per vintage | Key metrics |
|---|---|---|
| `lfs_layer3_score_drift` | 1 per (vintage_month, channel) | mean, std, PSI, KS, quantile cutoffs, baseline exceedance % |
| `lfs_layer3_feature_psi` | 11 per (vintage_month, channel) | PSI, mean/std shift per feature |
| `lfs_layer3_data_quality` | 1 per (vintage_month, channel) | Score missing rate, outlier rate |
| `lfs_layer3_feature_quality` | 11 per (vintage_month, channel) | Feature missing rate, outlier rate (baseline-derived bounds) |
| `lfs_layer3_feature_score_relationship` | 11 per (vintage_month, channel) | Pearson correlation: baseline, current, change |
| `lfs_layer3_population_mix` | N segments per (vintage_month, channel) | Segment %, monitors mix shift |

**Expected monitoring signals with this mock data:**
- `score_drift`: PSI ~ 0.05–0.15 (mild drift), KS > 0.05
- `feature_psi`: `feature_03` should show the highest PSI (~0.10–0.20)
- `population_mix`: source "organic" proportion should drop vs baseline

In [ ]:
l3_tables = build_layer3(enriched_df, baseline_df, config)
print(f"Layer 3 complete — {len(l3_tables)} tables produced:")
for tname, tdf in l3_tables.items():
    print(f"  {tname:<50}  {tdf.count():>3} rows  {len(tdf.columns)} cols")

### 6.1 Score Drift

One row per `(vintage_month, channel)`.  Key columns to watch:

- **`score_psi`** — PSI vs baseline. Flag if > 0.10 (yellow) or > 0.25 (red).
- **`score_ks`** — KS statistic. Larger = larger distributional shift.
- **`mean_lfs_score`** — Average score in current period.
- **`pct_accounts_above_p90_baseline`** — % of current accounts above the baseline p90 threshold.
  Rising value → score distribution shifting right.

In [ ]:
print("=== Score Drift (lfs_layer3_score_drift) ===\n")
(l3_tables["lfs_layer3_score_drift"]
 .select(
     "vintage_month", "channel",
     F.round("mean_lfs_score", 4).alias("mean_score"),
     F.round("std_lfs_score", 4).alias("std_score"),
     F.round("score_psi", 5).alias("psi"),
     F.round("score_ks", 5).alias("ks"),
     F.round("q50_score", 4).alias("q50"),
     F.round("q90_score", 4).alias("q90"),
     F.round("q95_score", 4).alias("q95"),
     F.round("mean_top10pct_score", 4).alias("top10_mean"),
     F.round("pct_accounts_above_p90_baseline", 4).alias("pct_above_p90_base"),
 )
 .orderBy("vintage_month", "channel")
 .show(truncate=False)
)

### 6.2 Feature PSI

One row per `(vintage_month, channel, feature_name)`.

- **`psi`** — Population Stability Index for this feature. `feature_03` should
  show the highest value given the +0.14 mean drift baked into the mock data.
- **`baseline_mean` / `current_mean`** — Direct comparison of mean values.
- **`baseline_std` / `current_std`** — Spread comparison.

In [ ]:
print("=== Feature PSI — sorted by PSI descending (2025-05, digital) ===\n")
(l3_tables["lfs_layer3_feature_psi"]
 .filter(
     (F.col("vintage_month") == "2025-05") &
     (F.col("channel") == "digital")
 )
 .select(
     "feature_name",
     F.round("psi", 5).alias("psi"),
     F.round("baseline_mean", 4).alias("base_mean"),
     F.round("current_mean", 4).alias("curr_mean"),
     (F.round("current_mean", 4) - F.round("baseline_mean", 4)).alias("mean_shift"),
     F.round("baseline_std", 4).alias("base_std"),
     F.round("current_std", 4).alias("curr_std"),
 )
 .orderBy(F.col("psi").desc())
 .show(truncate=False)
)

### 6.3 Data Quality

One row per `(vintage_month, channel)`.

- **`missing_score_rate`** — Fraction of accounts with NULL score.  Should be 0.0 with clean mock data.
- **`outlier_score_rate`** — Fraction with score outside [0, 1].  Should be 0.0 since mock scores are clamped.

In [ ]:
print("=== Data Quality (lfs_layer3_data_quality) ===\n")
(l3_tables["lfs_layer3_data_quality"]
 .select("vintage_month", "channel", "missing_score_rate", "outlier_score_rate")
 .orderBy("vintage_month", "channel")
 .show(truncate=False)
)

### 6.4 Feature-Score Relationship Drift

One row per `(vintage_month, channel, feature_name)`.

- **`baseline_corr`** — Pearson r with score in the baseline window.
- **`current_corr`** — Pearson r with score in the current period.
- **`corr_change`** — Signed difference.  Large negative = feature became less predictive.
- `None` values are expected for features with zero variance or fewer than 2 rows in a slice.

In [ ]:
print("=== Feature–Score Correlation Drift (2025-05, digital) ===\n")
(l3_tables["lfs_layer3_feature_score_relationship"]
 .filter(
     (F.col("vintage_month") == "2025-05") &
     (F.col("channel") == "digital")
 )
 .select(
     "feature_name",
     F.round("baseline_corr", 4).alias("base_corr"),
     F.round("current_corr", 4).alias("curr_corr"),
     F.round("corr_change", 4).alias("corr_chg"),
 )
 .orderBy(F.abs("corr_change").desc())
 .show(truncate=False)
)

### 6.5 Population Mix

One row per `(vintage_month, channel, segment_type, segment_value)`.
Tracks composition shifts across four segment dimensions: source, line,
receivable bucket, saleamount bucket.

Watch for shifts in `pct_of_channel_accounts` vs the overall baseline mix
(run the same query on `baseline_df` to compare).

In [ ]:
print("=== Population Mix — source segment (2025-05) ===\n")
(l3_tables["lfs_layer3_population_mix"]
 .filter(
     (F.col("vintage_month") == "2025-05") &
     (F.col("segment_type") == "source")
 )
 .select(
     "channel", "segment_type", "segment_value",
     "account_count",
     F.round("pct_of_channel_accounts", 4).alias("pct"),
 )
 .orderBy("channel", "segment_value")
 .show(truncate=False)
)

print("=== Population Mix — receivable_bucket segment (2025-05) ===\n")
(l3_tables["lfs_layer3_population_mix"]
 .filter(
     (F.col("vintage_month") == "2025-05") &
     (F.col("segment_type") == "receivable_bucket")
 )
 .select(
     "channel", "segment_value",
     "account_count",
     F.round("pct_of_channel_accounts", 4).alias("pct"),
 )
 .orderBy("channel", "segment_value")
 .show(truncate=False)
)

## 7. Quick Validation Checks

These inline assertions replicate the checks from `docs/validation_checklist.md`.
Run this cell to confirm all outputs are internally consistent before promoting
to production.

In [ ]:
from pyspark.sql import Window

print("Running validation checks...\n")
errors = []

# 1. Layer 1 row count matches current_df.
l1_count = enriched_df.count()
src_count = current_df.count()
if l1_count != src_count:
    errors.append(f"[FAIL] Layer1 row count {l1_count} != source {src_count}")
else:
    print(f"[OK]   Layer1 row count matches source: {l1_count:,}")

# 2. Layer2 account_count sums to Layer1 total.
l2_total = (
    l2_tables["lfs_layer2_overall_summary"]
    .agg(F.sum("account_count").alias("total"))
    .collect()[0]["total"]
)
if l2_total != l1_count:
    errors.append(f"[FAIL] Layer2 account_count sum {l2_total} != Layer1 {l1_count}")
else:
    print(f"[OK]   Layer2 account_count sums to Layer1 total: {l2_total:,}")

# 3. pct_of_channel_vintage_accounts sums to 1.0 per (vintage_month, channel).
w = Window.partitionBy("vintage_month", "channel")
bad_pct = (
    l2_tables["lfs_layer2_score_distribution"]
    .groupBy("vintage_month", "channel")
    .agg(F.sum("pct_of_channel_vintage_accounts").alias("total_pct"))
    .filter(F.abs(F.col("total_pct") - 1.0) > 1e-9)
    .count()
)
if bad_pct > 0:
    errors.append(f"[FAIL] {bad_pct} partition(s) where pct != 1.0")
else:
    print("[OK]   pct_of_channel_vintage_accounts sums to 1.0 in all partitions")

# 4. Both channels present in every Layer3 table.
for tname, tdf in l3_tables.items():
    if "channel" in tdf.columns:
        channels = {r["channel"] for r in tdf.select("channel").distinct().collect()}
        if channels != {"digital", "directmail"}:
            errors.append(f"[FAIL] {tname} missing channels: {channels}")

print("[OK]   Both channels present in all Layer3 tables")

# 5. PSI and KS are non-negative.
neg_psi = l3_tables["lfs_layer3_score_drift"].filter(F.col("score_psi") < 0).count()
neg_ks  = l3_tables["lfs_layer3_score_drift"].filter(F.col("score_ks") < 0).count()
if neg_psi or neg_ks:
    errors.append(f"[FAIL] Negative PSI={neg_psi} or KS={neg_ks}")
else:
    print("[OK]   PSI and KS are non-negative")

# 6. population_mix pct sums to 1.0 per (vintage_month, channel, segment_type).
bad_mix = (
    l3_tables["lfs_layer3_population_mix"]
    .groupBy("vintage_month", "channel", "segment_type")
    .agg(F.sum("pct_of_channel_accounts").alias("total_pct"))
    .filter(F.abs(F.col("total_pct") - 1.0) > 1e-9)
    .count()
)
if bad_mix > 0:
    errors.append(f"[FAIL] {bad_mix} population_mix partition(s) where pct != 1.0")
else:
    print("[OK]   population_mix pct sums to 1.0 in all partitions")

# 7. segment_value is always a string.
non_str = [
    r["segment_value"]
    for r in l3_tables["lfs_layer3_population_mix"].select("segment_value").collect()
    if not isinstance(r["segment_value"], str)
]
if non_str:
    errors.append(f"[FAIL] Non-string segment_value found: {non_str[:3]}")
else:
    print("[OK]   All segment_value entries are strings")

print()
if errors:
    print("VALIDATION FAILED:")
    for e in errors:
        print(" ", e)
else:
    print("All validation checks passed.")

## 8. Next Steps

### Connecting to real data

Replace the mock-data cells with production source reads.
The simplest notebook workflow once your environment is configured:

```python
from framework.runner import run

outputs = run(
    model="lfs",
    score_month="2025-05",
    model_version="v1.0",
    spark=spark,
    write_output=False,    # dry-run first
    return_outputs=True,
)

# Inspect outputs here, then re-run with write_output=True to promote.
```

### Things to verify on first real run

1. Row counts match the source table exactly.
2. Both channels have accounts in every vintage.
3. `dq_missing_flag` rate is below your SLA threshold.
4. `score_psi` is near zero for months within the baseline window.
5. `feature_03` (or whichever features are known to drift) shows expected PSI.

### Adding a second model

Drop a new YAML file in `conf/models/` and call `run(model="new_model", ...)`.
The framework requires no code changes.

See `docs/validation_checklist.md` for the full SQL-based QA checklist.